# Filtragem espacial e correlação 2D

Neste notebook, relacionamos vizinhanças, kernels, suavização, realce e detecção de bordas por exemplos numéricos e visuais.

## Objetivos

- calcular uma correlação 2D e interpretar sua política de borda;
- distinguir correlação de convolução com um kernel assimétrico;
- comparar média, Gauss, mediana, sharpening, Sobel, Laplaciano e Canny.

## 1. Instalação

A instalação usa a versão commitada na branch principal do repositório.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.feature_extractor import FeatureExtractor
from dip_toolkit.modules.image_loader import ImageLoader
from dip_toolkit.modules.image_preprocessor import ImagePreprocessor
from dip_toolkit.modules.visualization import Visualization

filters = ImagePreprocessor()
features = FeatureExtractor()
loader = ImageLoader()
visualization = Visualization()

## 2. Vizinhança e kernel

A vizinhança é o conjunto local de pixels observado ao redor de uma posição. Um **kernel** é uma pequena matriz de pesos, normalmente com dimensões ímpares para possuir um centro definido. Na correlação, alinhamos o centro do kernel ao pixel, multiplicamos os valores correspondentes e somamos os produtos.

In [ ]:
small_image = np.arange(1, 10, dtype=np.float64).reshape(3, 3)
asymmetric_kernel = np.array([[1, 2, 3], [0, 0, 0], [-1, -2, -3]])
center_manual = np.sum(small_image * asymmetric_kernel)
correlated = filters.correlate(small_image, asymmetric_kernel)
center_manual, correlated

### Política de borda e contrato numérico

`correlate` aceita grayscale 2D e kernels NumPy 2D, finitos, não vazios e ímpares. A saída preserva o shape, usa `float64` e não é normalizada nem limitada. Fora da imagem, os pixels valem zero. As entradas não são alteradas.

## 3. Correlação não é convolução

**Correlação:** aplica o kernel como fornecido.

**Convolução:** inverte o kernel horizontal e verticalmente antes da aplicação.

Kernels simétricos escondem a diferença. Para demonstrar convolução, invertemos o kernel e reutilizamos `correlate`; não criamos uma segunda API.

In [ ]:
flipped_kernel = np.flip(asymmetric_kernel, axis=(0, 1))
convolved_for_demo = filters.correlate(small_image, flipped_kernel)
correlated[1, 1], convolved_for_demo[1, 1], flipped_kernel

## 4. Identidade e média

A identidade seleciona o pixel central. Na média, a normalização é explícita e responsabilidade de quem constrói o kernel.

In [ ]:
identity_kernel = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])
mean_kernel = np.ones((3, 3), dtype=np.float64) / 9
identity_result = filters.correlate(small_image, identity_kernel)
mean_result = filters.correlate(small_image, mean_kernel)
np.array_equal(identity_result, small_image), mean_result[1, 1], mean_result[0, 0]

## 5. Suavização, realce e bordas

Média, Gauss, mediana e sharpening aceitam grayscale ou três canais e estendem a borda. As operações lineares retornam `float64` sem clipping; mediana preserva dtype. Sobel e Laplaciano aceitam grayscale e preservam a resposta numérica. Canny exige grayscale `uint8` e retorna 0 ou 255.

In [ ]:
impulse = np.zeros((7, 7), dtype=np.uint8)
impulse[3, 3] = 255
step = np.zeros((7, 9), dtype=np.uint8)
step[:, 4:] = 255
mean_impulse = filters.mean_filter(impulse)
gaussian_impulse = filters.gaussian_filter(impulse, sigma=1.0)
median_impulse = filters.median_filter(impulse)
sharpened_impulse = filters.sharpen(impulse)
sobel_step = features.sobel(step)
laplacian_step = features.laplacian(step)
canny_step = features.canny(step, 50, 150)

In [ ]:
figure, axes = plt.subplots(2, 4, figsize=(14, 7))
synthetic_results = [
    (impulse, "Impulso"),
    (mean_impulse, "Média"),
    (gaussian_impulse, "Gauss"),
    (median_impulse, "Mediana"),
    (step, "Degrau"),
    (sobel_step, "Sobel"),
    (np.abs(laplacian_step), "|Laplaciano|"),
    (canny_step, "Canny"),
]
for axis, (result, title) in zip(axes.ravel(), synthetic_results, strict=True):
    axis.imshow(result, cmap="gray")
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## 6. Aplicação em imagem real

A imagem é obtida por `download_course_image` e aberta em grayscale pelo `ImageLoader`; o processamento permanece separado da visualização.

In [ ]:
image_path = download_course_image("cameraman_original.png")
image_gray = loader.load_image(image_path, flags=cv.IMREAD_GRAYSCALE)
mean_gray = filters.mean_filter(image_gray, 3)
gaussian_gray = filters.gaussian_filter(image_gray, 5, 1.2)
median_gray = filters.median_filter(image_gray, 3)
sharpened_gray = filters.sharpen(image_gray, 0.5)
sobel_gray = features.sobel(image_gray)
laplacian_gray = features.laplacian(image_gray)
canny_gray = features.canny(image_gray, 80, 160)

In [ ]:
def display_uint8(response):
    magnitude = np.abs(response)
    if magnitude.max() == 0:
        return np.zeros(response.shape, dtype=np.uint8)
    return np.rint(magnitude / magnitude.max() * 255).astype(np.uint8)


visualization.compare_images(
    [
        image_gray,
        np.clip(mean_gray, 0, 255).astype(np.uint8),
        np.clip(gaussian_gray, 0, 255).astype(np.uint8),
        median_gray,
        np.clip(sharpened_gray, 0, 255).astype(np.uint8),
    ],
    ["Original", "Média", "Gauss", "Mediana", "Sharpening"],
)
plt.show()
visualization.compare_images(
    [image_gray, display_uint8(sobel_gray), display_uint8(laplacian_gray), canny_gray],
    ["Original", "Sobel", "|Laplaciano|", "Canny"],
)
plt.show()

## Exercício final

1. Crie outro kernel assimétrico 3x3 e calcule manualmente o pixel central.
2. Compare a correlação com o kernel invertido nos dois eixos.
3. Teste tamanhos 3, 5 e 7 na média e no Gauss.
4. Ajuste os limiares do Canny e compare com Sobel e Laplaciano.
5. Explique por que respostas assinadas não devem virar `uint8` diretamente.

In [ ]:
# TODO: implemente o kernel assimétrico e registre suas conclusões.
student_kernel = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])